In [2]:
import torch

In [5]:
a = torch.zeros(2,3)
print("a: \n", a)

b = torch.ones([2,3])
print("b: \n", b)

c = torch.rand((2,3))
print("c: \n", c)

a: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
b: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
c: 
 tensor([[0.8207, 0.1044, 0.5107],
        [0.3239, 0.3607, 0.1266]])



# Autograd computation : A small example

In [12]:
BATCH_SIZE = 16
INPUT_SIZE = 1000
HIDDEN_SIZE = 100
OUTPUT_SIZE = 10 

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()

        self.layer1 = torch.nn.Linear(1000,100)
        self.relu   = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(100,10) 

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x) 

        return x

some_input = torch.rand(BATCH_SIZE , INPUT_SIZE, requires_grad = True) 
ideal_output = torch.rand(BATCH_SIZE, OUTPUT_SIZE, requires_grad = False)

model = TinyModel()


In [51]:
#Constants 

input_tensor = torch.unsqueeze(torch.tensor([1, 0, 1, 0 , 1, 1, 0, 1, 1, 1], dtype = float) , 0)

print("input_tensor shape", input_tensor.size())
print("input_tensor : \n" , input_tensor) 
weight_matrix = torch.rand((10,2) , dtype = float)
print("weight matrix : \n", weight_matrix)


y_true = torch.tensor([1, 0]) 
print("y_true : ", y_true)
y_pred = torch.squeeze(torch.matmul(input_tensor, weight_matrix), 0)
print("y_pred shape", y_pred.size()) 
print("y_pred iteration_1 : ", y_pred)
print()

error_1 = ((y_true[0] - y_pred[0])**2 + (y_true[1] - y_pred[1])**2 ) / 2
print("error 1 ", error_1)

input_tensor shape torch.Size([1, 10])
input_tensor : 
 tensor([[1., 0., 1., 0., 1., 1., 0., 1., 1., 1.]], dtype=torch.float64)
weight matrix : 
 tensor([[0.1555, 0.5518],
        [0.1344, 0.8719],
        [0.7431, 0.6012],
        [0.9479, 0.9600],
        [0.5439, 0.9675],
        [0.0911, 0.7084],
        [0.2318, 0.4510],
        [0.1819, 0.6513],
        [0.9049, 0.1394],
        [0.3129, 0.8420]], dtype=torch.float64)
y_true :  tensor([1, 0])
y_pred shape torch.Size([2])
y_pred iteration_1 :  tensor([2.9332, 4.4616], dtype=torch.float64)

error 1  tensor(11.8214, dtype=torch.float64)


In [88]:
def details(y , tensor_name):
    print(f"{tensor_name}          : " , y)
    print(f"{tensor_name}.shape    : ", y.shape)
    print(f"{tensor_name}.datatype : ", y.dtype)
    print("\n")
    

batch_size = 16
input_size = 3
hidden_size= 10
output_size= 2

class small_model(torch.nn.Module):
    def __init__(self):
        super(small_model, self).__init__()
        self.layer1  = torch.nn.Linear(input_size, hidden_size)
        self.relu_fn = torch.nn.ReLU()
        self.layer2  = torch.nn.Linear(hidden_size, output_size)

    def forward(self , x):
        x = self.layer1(x)
        # print("post Linear layer 1 : \n", x) 
        x = self.relu_fn(x)
        # print("post Relu : \n", x)
        x = self.layer2(x)
        
        return x

torch.manual_seed(62)
input_tensor = torch.rand([1,input_size])
details(input_tensor, "input_tensor")

ideal_output = torch.rand([1, output_size])
details(ideal_output, "ideal_output")

model1 = small_model()
print("model parameters")
for param in model1.parameters():
    print(param)
print("--------------------------------------")
print("Layer 1 weights :", model1.layer1.weight) #Note the size of W matrix . PyTorch stores W with size(ouput, input) 
print("Layer 1 bias    :", model1.layer1.bias)
print("\n")
print("Layer 2 weights : ",model1.layer2.weight)
print("Layer 2 bias    : ",model1.layer2.bias)
print("\n")
print("--------------------------------------")
prediction = model1(input_tensor)
print("prediction : \n", prediction)


input_tensor          :  tensor([[0.6412, 0.8290, 0.2115]])
input_tensor.shape    :  torch.Size([1, 3])
input_tensor.datatype :  torch.float32


ideal_output          :  tensor([[0.5220, 0.5978]])
ideal_output.shape    :  torch.Size([1, 2])
ideal_output.datatype :  torch.float32


model parameters
Parameter containing:
tensor([[ 0.1558, -0.2091,  0.0414],
        [ 0.1644,  0.4873, -0.0214],
        [-0.1116,  0.5276,  0.2503],
        [-0.0988,  0.4927,  0.0124],
        [ 0.4716, -0.4791,  0.4959],
        [-0.1992,  0.0200,  0.0825],
        [ 0.5337, -0.5679,  0.4291],
        [-0.3351, -0.1892,  0.3663],
        [-0.4621, -0.4768,  0.0816],
        [-0.3837,  0.0525, -0.2330]], requires_grad=True)
Parameter containing:
tensor([ 0.5251, -0.0409,  0.1237, -0.3090,  0.4158, -0.2715,  0.1473,  0.1360,
         0.3459, -0.0321], requires_grad=True)
Parameter containing:
tensor([[ 2.0813e-01, -2.4850e-02,  1.7990e-01, -8.3261e-02, -1.9818e-01,
         -1.9542e-01, -1.7922e-01, -2.0841e

In [85]:
optimizer = torch.optim.SGD(model1.parameters() , lr = 0.001)
loss      = (ideal_output - prediction).pow(2).mean()
print("loss :", loss)

loss.backward() #This only calcuales the gradients BUT DOES NOT update the weights

loss : tensor(0.2579, grad_fn=<MeanBackward0>)


In [65]:
print("Layer 1 weights :", model1.layer1.weight)#Note the size of W matrix . PyTorch stores W with size(ouput, input) 
print("Layer 1 weights gradients: \n", model1.layer1.weight.grad)
print("Layer 1 bias    :", model1.layer1.bias)
print("Layer 1 bias gradients  : \n", model1.layer1.bias.grad)

Layer 1 weights : Parameter containing:
tensor([[ 0.0916,  0.1710, -0.5509],
        [ 0.1003, -0.2018,  0.5176],
        [ 0.0248,  0.4365,  0.2601],
        [ 0.2228, -0.5475,  0.4988],
        [-0.4721,  0.2513,  0.2622],
        [-0.0038,  0.4974, -0.1528],
        [-0.3408,  0.1901, -0.1286],
        [-0.4865, -0.4103, -0.3685],
        [ 0.1662, -0.4037,  0.2633],
        [ 0.4465, -0.2342,  0.3265]], requires_grad=True)
Layer 1 weights gradients: 
 tensor([[0.0000, 0.0000, 0.0000],
        [0.6115, 0.1900, 0.2891],
        [0.4649, 0.1444, 0.2198],
        [0.0000, 0.0000, 0.0000],
        [0.2062, 0.0641, 0.0975],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000]])
Layer 1 bias    : Parameter containing:
tensor([ 0.0121,  0.3673, -0.0733, -0.3605,  0.4363, -0.3550,  0.1341,  0.3290,
        -0.4179, -0.5248], requires_grad=True)
Layer 1 bias gradients  : 
 tens

In [66]:
optimizer.step()
print("Layer 1 weights :", model1.layer1.weight)
print("Layer 1 weights gradients: \n", model1.layer1.weight.grad)
print("Layer 1 bias    :", model1.layer1.bias)
print("Layer 1 bias gradients  : \n", model1.layer1.bias.grad)

Layer 1 weights : Parameter containing:
tensor([[ 0.0916,  0.1710, -0.5509],
        [ 0.0997, -0.2020,  0.5173],
        [ 0.0243,  0.4364,  0.2598],
        [ 0.2228, -0.5475,  0.4988],
        [-0.4724,  0.2512,  0.2621],
        [-0.0038,  0.4974, -0.1528],
        [-0.3408,  0.1901, -0.1286],
        [-0.4865, -0.4103, -0.3685],
        [ 0.1662, -0.4037,  0.2633],
        [ 0.4465, -0.2342,  0.3265]], requires_grad=True)
Layer 1 weights gradients: 
 tensor([[0.0000, 0.0000, 0.0000],
        [0.6115, 0.1900, 0.2891],
        [0.4649, 0.1444, 0.2198],
        [0.0000, 0.0000, 0.0000],
        [0.2062, 0.0641, 0.0975],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000]])
Layer 1 bias    : Parameter containing:
tensor([ 0.0121,  0.3666, -0.0739, -0.3605,  0.4361, -0.3550,  0.1341,  0.3290,
        -0.4179, -0.5248], requires_grad=True)
Layer 1 bias gradients  : 
 tens

In [89]:
optimizer = torch.optim.SGD(model1.parameters() , lr = 0.001)
for i in range(0,1000) :
    # print(f"Iteration : {i}")
    prediction = model1(input_tensor)
    loss      = (ideal_output - prediction).pow(2).mean()
    if i % 10 == 0 :
        print(f" Iteration : {i}   |loss :", loss)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # print("\n")

 Iteration : 0   |loss : tensor(0.2579, grad_fn=<MeanBackward0>)
 Iteration : 10   |loss : tensor(0.2438, grad_fn=<MeanBackward0>)
 Iteration : 20   |loss : tensor(0.2303, grad_fn=<MeanBackward0>)
 Iteration : 30   |loss : tensor(0.2176, grad_fn=<MeanBackward0>)
 Iteration : 40   |loss : tensor(0.2055, grad_fn=<MeanBackward0>)
 Iteration : 50   |loss : tensor(0.1940, grad_fn=<MeanBackward0>)
 Iteration : 60   |loss : tensor(0.1831, grad_fn=<MeanBackward0>)
 Iteration : 70   |loss : tensor(0.1728, grad_fn=<MeanBackward0>)
 Iteration : 80   |loss : tensor(0.1630, grad_fn=<MeanBackward0>)
 Iteration : 90   |loss : tensor(0.1538, grad_fn=<MeanBackward0>)
 Iteration : 100   |loss : tensor(0.1450, grad_fn=<MeanBackward0>)
 Iteration : 110   |loss : tensor(0.1367, grad_fn=<MeanBackward0>)
 Iteration : 120   |loss : tensor(0.1288, grad_fn=<MeanBackward0>)
 Iteration : 130   |loss : tensor(0.1214, grad_fn=<MeanBackward0>)
 Iteration : 140   |loss : tensor(0.1144, grad_fn=<MeanBackward0>)
 Itera

In [90]:
print(ideal_output)
print(prediction)

tensor([[0.5220, 0.5978]])
tensor([[0.5090, 0.5718]], grad_fn=<AddmmBackward0>)


The Loss : 

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$